# 01 — Ingestion
Reads manually uploaded files from Unity Catalog volumes and writes them to the raw Delta layer.

**Catalog:** `airbnb_app`  
**Layer:** `raw`  

| Section | Source | Granularity | Output tables |
|---------|--------|-------------|---------------|
| A — Inside Airbnb | insideairbnb.com | Listing / neighbourhood | `airbnb_app.raw.airbnb_{file_type}_{city}` |
| B — House prices | ONS HPSSA | MSOA | `airbnb_app.raw.house_prices_msoa` |
| C — Amenities | ONS / Ordnance Survey | LAD / MSOA | `airbnb_app.raw.amenities_{type}` |
| D — Private rents | ONS PIPR | LAD / Region | `airbnb_app.raw.rent_data` |

---

### Before running — confirm files are uploaded to the correct volumes

```
/Volumes/airbnb_app/raw/
├── london/london/
│   ├── listings.csv.gz
│   ├── calendar.csv.gz
│   ├── reviews.csv.gz
│   ├── neighbourhoods.csv
│   └── neighbourhoods.geojson
├── manchester/manchester/  (same structure)
├── edinburgh/edinburgh/    (same structure)
├── bristol/bristol/        (same structure)
├── house_prices/
│   └── medianpricepaidmsoa.xlsx
├── amenities_data/
│   ├── accesstoamenitiesgpsurgeries.xlsx
│   ├── accesstoamenitiesparksandplayareas.xlsx
│   └── accesstoamenitiestraveltimetorailstations.xlsx
└── rent_data/
    └── priceindexofprivaterentsukmonthlypricestatistics.xlsx
```

> **Note:** Rental and House Price data does not include Edinburgh. Edinburgh rental estimates must be sourced separately from the Scottish Government private sector rent statistics.

## 0. Config

In [0]:
RAW_DB = "airbnb_app.raw"

# ── Inside Airbnb ─────────────────────────────────────────────────────────────
AIRBNB_PATH = "/Volumes/airbnb_app/raw"

FILE_MANIFEST = {
    "london":     ["listings", "calendar", "reviews", "neighbourhoods", "neighbourhoods_geo"],
    "manchester": ["listings", "calendar", "reviews", "neighbourhoods", "neighbourhoods_geo"],
    "edinburgh":  ["listings", "calendar", "reviews", "neighbourhoods", "neighbourhoods_geo"],
    "bristol":    ["listings", "calendar", "reviews", "neighbourhoods", "neighbourhoods_geo"],
}

FILENAME_MAP = {
    "listings":           "listings",
    "calendar":           "calendar",
    "reviews":            "reviews",
    "neighbourhoods":     "neighbourhoods",
    "neighbourhoods_geo": "neighbourhoods",
}

GEOJSON_FILES = {"neighbourhoods_geo"}

CSV_READ_OPTIONS = {
    "header":      "true",
    "inferSchema": "true",
    "escape":      '"',
    "multiLine":   "true",
    "encoding":    "UTF-8",
}

# ── House prices ──────────────────────────────────────────────────────────────
# ONS HPSSA — England and Wales only, MSOA level
# Edinburgh requires separate data from Registers of Scotland (ros.gov.uk)
HOUSE_PRICE_PATH   = "/Volumes/airbnb_app/raw/house_prices"
HOUSE_PRICE_FILE   = "medianpricepaidmsoa.xlsx"
HOUSE_PRICE_SHEET  = 2      # third sheet (0-indexed)
HOUSE_PRICE_SKIP   = 2      # skip 2 metadata rows
MOST_RECENT_PERIOD = "year_ending_sep_2025"  # after column rename

# ── Amenities ─────────────────────────────────────────────────────────────────
# ONS Access to Amenities — England and Wales only (LAD level)
# Rail stations file is at MSOA level on a different sheet
# Edinburgh will have no rows — noted as data gap in assumptions
AMENITIES_PATH = "/Volumes/airbnb_app/raw/amenities_data"

AMENITIES_FILES = {
    "gp_surgeries":  "accesstoamenitiesgpsurgeries.xlsx",
    "parks":         "accesstoamenitiesparksandplayareas.xlsx",
    "rail_stations": "accesstoamenitiestraveltimetorailstations.xlsx",
}

# Rail stations uses a different sheet (9th, index 8) and is MSOA level
# GP surgeries and parks use the 4th sheet (index 3) and are LAD level
AMENITIES_SHEET = {
    "gp_surgeries":  3,
    "parks":         3,
    "rail_stations": 8,
}

AMENITIES_SKIPROWS = 5  # all three files: 5 metadata rows before header

# ── Rent data ─────────────────────────────────────────────────────────────────
# ONS PIPR — UK-wide including Scotland EXCEPT Edinburgh not present
# Edinburgh rental data must be sourced from Scottish Government statistics
RENT_PATH     = "/Volumes/airbnb_app/raw/rent_data"
RENT_FILE     = "priceindexofprivaterentsukmonthlypricestatistics.xlsx"
RENT_SHEET    = 3   # 'Table 1' sheet (0-indexed)
RENT_SKIPROWS = 2   # skip 2 metadata rows

## 1. Setup

In [0]:
%pip install openpyxl

In [0]:
import os
import gzip
import shutil
import pandas as pd
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RAW_DB}")
print(f"Schema ready: {RAW_DB}")

ingest_log = []

## 2. Helpers

In [0]:
def resolve_path(city: str, file_type: str) -> str:
    """
    Resolves the volume path for a given city and file type.
    Databricks creates a double city subfolder on upload: /london/london/
    Checks .csv.gz first, falls back to .csv if already decompressed.
    """
    base     = f"{AIRBNB_PATH}/{city}/{city}"
    filename = FILENAME_MAP[file_type]

    if file_type == "neighbourhoods_geo":
        return f"{base}/{filename}.geojson"
    if file_type == "neighbourhoods":
        return f"{base}/{filename}.csv"

    gz_path  = f"{base}/{filename}.csv.gz"
    csv_path = f"{base}/{filename}.csv"
    if os.path.exists(gz_path):
        return gz_path
    elif os.path.exists(csv_path):
        return csv_path
    else:
        return gz_path


def check_file(path: str) -> bool:
    """Check a file exists and print its size. Returns False if missing."""
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1_048_576
        print(f"  ✓ {os.path.basename(path)} ({size_mb:.1f} MB)")
        return True
    else:
        print(f"  ✗ NOT FOUND: {path}")
        return False


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename DataFrame columns to be Delta-compatible.
    Drops columns with null names, removes spaces and special characters.
    """
    df = df.loc[:, df.columns.notna()]  # drop null column names
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(".", "", regex=False)
    )
    return df


def write_raw_spark(df, table: str) -> int:
    """Write a Spark DataFrame to a raw Delta table."""
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    count = df.count()
    print(f"  ✓ {table} ({count:,} rows)")
    return count


def write_raw_pandas(df_pd: pd.DataFrame, table: str, source: str) -> int:
    """Convert a pandas DataFrame to Spark and write to a raw Delta table."""
    df_spark = spark.createDataFrame(df_pd)
    df_spark = (
        df_spark
        .withColumn("_source",      F.lit(source))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    return write_raw_spark(df_spark, table)


print("Helpers loaded.")

---
# Part A — Inside Airbnb

Five files per city across four cities — 20 tables total.

| File | Description | Format |
|------|-------------|--------|
| listings | One row per listing — pricing, location, host info, availability, review scores | `.csv.gz` |
| calendar | One row per listing per day for 365 days — availability | `.csv.gz` |
| reviews | One row per guest review — reviewer details and comment text | `.csv.gz` |
| neighbourhoods | Reference list of neighbourhood names per city | `.csv` |
| neighbourhoods_geo | GeoJSON boundary polygons for each neighbourhood | `.geojson` |

## A1. Check Airbnb files exist

In [0]:
missing_files = []
found_files   = []

for city, file_types in FILE_MANIFEST.items():
    for file_type in file_types:
        path = resolve_path(city, file_type)
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / 1_048_576
            found_files.append({"city": city, "file_type": file_type, "path": path, "size_mb": round(size_mb, 2)})
        else:
            missing_files.append({"city": city, "file_type": file_type, "expected_path": path})

print(f"Found:   {len(found_files)} files")
print(f"Missing: {len(missing_files)} files")

if missing_files:
    print("\nMissing files — upload before continuing:")
    display(pd.DataFrame(missing_files))
else:
    print("\nAll Airbnb files found.")
    display(pd.DataFrame(found_files))

## A2. Decompress .gz files

In [0]:
for entry in found_files:
    if entry["path"].endswith(".gz"):
        gz_path  = entry["path"]
        csv_path = gz_path.replace(".csv.gz", ".csv")
        if not os.path.exists(csv_path):
            print(f"Decompressing {entry['city']}/{entry['file_type']} ...")
            with gzip.open(gz_path, "rb") as gz_in:
                with open(csv_path, "wb") as csv_out:
                    shutil.copyfileobj(gz_in, csv_out)
            print(f"  Done — {os.path.getsize(csv_path)/1_048_576:.1f} MB")
        else:
            print(f"Already decompressed: {entry['city']}/{entry['file_type']}")
        entry["path"] = csv_path

print("\nDecompression complete.")

## A3. Write Airbnb files to Delta

In [0]:
for entry in found_files:
    city      = entry["city"]
    file_type = entry["file_type"]
    src_path  = entry["path"]
    table     = f"{RAW_DB}.airbnb_{file_type}_{city}"

    try:
        if file_type in GEOJSON_FILES:
            df = spark.read.text(src_path)
            df = df.withColumnRenamed("value", "geojson_raw")
        else:
            df = spark.read.csv(src_path, **CSV_READ_OPTIONS)

        df = (
            df
            .withColumn("_city",        F.lit(city))
            .withColumn("_file_type",   F.lit(file_type))
            .withColumn("_ingested_at", F.current_timestamp())
        )

        count = write_raw_spark(df, table)
        ingest_log.append({"section": "airbnb", "dataset": f"{file_type}_{city}", "status": "ok", "rows": count})

    except Exception as e:
        print(f"  ✗ {table} — {e}")
        ingest_log.append({"section": "airbnb", "dataset": f"{file_type}_{city}", "status": "error", "error": str(e)})

## A4. Airbnb spot checks

In [0]:
# Row counts across all cities and file types
spark.sql("""
    SELECT _city, _file_type, COUNT(*) AS row_count, MAX(_ingested_at) AS ingested_at
    FROM (
        SELECT _city, _file_type, _ingested_at FROM airbnb_app.raw.airbnb_listings_london
        UNION ALL SELECT _city, _file_type, _ingested_at FROM airbnb_app.raw.airbnb_listings_manchester
        UNION ALL SELECT _city, _file_type, _ingested_at FROM airbnb_app.raw.airbnb_listings_edinburgh
        UNION ALL SELECT _city, _file_type, _ingested_at FROM airbnb_app.raw.airbnb_listings_bristol
    )
    GROUP BY _city, _file_type
    ORDER BY _city, _file_type
""").display()

In [0]:
# Sample listings — check key columns
spark.sql("""
    SELECT id, name, neighbourhood_cleansed, room_type, price,
           minimum_nights, number_of_reviews, latitude, longitude
    FROM airbnb_app.raw.airbnb_listings_london
    LIMIT 10
""").display()

In [0]:
# Sample reviews — check multiline text handled correctly
spark.sql("""
    SELECT listing_id, date, LEFT(comments, 200) AS comment_preview
    FROM airbnb_app.raw.airbnb_reviews_london
    WHERE comments IS NOT NULL
    LIMIT 5
""").display()

---
# Part B — House Prices

**Source:** ONS House Price Statistics for Small Areas (HPSSA)  
**Granularity:** MSOA  
**Coverage:** England and Wales only  

> Edinburgh is not covered — house price data for Edinburgh must be sourced separately from the Registers of Scotland (`ros.gov.uk`).  
> Raw file ingested as-is in wide format. Reshaping to one row per MSOA with the most recent price only happens in the cleaning notebook.

## B1. Check house price file exists

In [0]:
hp_file_path = f"{HOUSE_PRICE_PATH}/{HOUSE_PRICE_FILE}"
print("Checking house price file...")
if not check_file(hp_file_path):
    raise FileNotFoundError(f"File not found: {hp_file_path}")

## B2. Ingest house prices to Delta

In [0]:
HP_TABLE = f"{RAW_DB}.house_prices_msoa"
print("Ingesting house prices...")

try:
    df_hp = pd.read_excel(
        hp_file_path,
        sheet_name=HOUSE_PRICE_SHEET,
        skiprows=HOUSE_PRICE_SKIP,
        dtype=str,
        engine="openpyxl",
    )
    df_hp = df_hp.dropna(how="all")
    df_hp = clean_column_names(df_hp)

    print(f"  Shape: {df_hp.shape}")
    print(f"  Sample columns: {list(df_hp.columns[:6])}")

    count = write_raw_pandas(df_hp, HP_TABLE, source="ONS HPSSA MSOA")
    ingest_log.append({"section": "house_prices", "dataset": "house_prices_msoa", "status": "ok", "rows": count})

except Exception as e:
    print(f"  ✗ {HP_TABLE} — {e}")
    ingest_log.append({"section": "house_prices", "dataset": "house_prices_msoa", "status": "error", "error": str(e)})

## B3. House prices spot check

In [0]:
spark.sql(f"""
    SELECT local_authority_code, local_authority_name,
           msoa_code, msoa_name,
           {MOST_RECENT_PERIOD}
    FROM {HP_TABLE}
    LIMIT 10
""").display()

---
# Part C — Amenities Data

Three files from the ONS Access to Amenities dataset.

| File | Granularity | Sheet | Description |
|------|-------------|-------|-------------|
| `accesstoamenitiesgpsurgeries.xlsx` | LAD | Sheet 4 (index 3) | Count of GP surgeries per LAD |
| `accesstoamenitiesparksandplayareas.xlsx` | LAD | Sheet 4 (index 3) | Count of parks, play areas, playing fields per LAD |
| `accesstoamenitiestraveltimetorailstations.xlsx` | MSOA | Sheet 9 (index 8) | % population within 15/30/60 min walk of rail station |

All files: 5 metadata rows to skip, header on row 6.

> **Coverage:** England and Wales only — Edinburgh will have no rows in GP surgeries and parks tables. Rail stations is at MSOA level.

## C1. Check amenities files exist

In [0]:
print("Checking amenities files...")
amenities_ok = True
for name, filename in AMENITIES_FILES.items():
    path = f"{AMENITIES_PATH}/{filename}"
    if not check_file(path):
        amenities_ok = False

if not amenities_ok:
    raise FileNotFoundError("One or more amenities files missing — upload before continuing.")

## C2. Inspect amenities structure

In [0]:
# Confirm columns before full ingest
for name, filename in AMENITIES_FILES.items():
    path = f"{AMENITIES_PATH}/{filename}"
    df_preview = pd.read_excel(
        path,
        sheet_name=AMENITIES_SHEET[name],
        skiprows=AMENITIES_SKIPROWS,
        nrows=2,
        dtype=str,
        engine="openpyxl",
    )
    df_preview = df_preview.loc[:, df_preview.columns.notna()]
    print(f"\n{name}: {list(df_preview.columns)}")
    display(df_preview)

## C3. Ingest amenities files to Delta

In [0]:
for name, filename in AMENITIES_FILES.items():
    path  = f"{AMENITIES_PATH}/{filename}"
    table = f"{RAW_DB}.amenities_{name}"
    print(f"\nIngesting {name}...")

    try:
        df = pd.read_excel(
            path,
            sheet_name=AMENITIES_SHEET[name],
            skiprows=AMENITIES_SKIPROWS,
            dtype=str,
            engine="openpyxl",
        )
        df = df.dropna(how="all")
        df = clean_column_names(df)   # drops null columns + renames
        df["amenity_type"] = name

        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")

        count = write_raw_pandas(df, table, source=f"ONS Access to Amenities — {name}")
        ingest_log.append({"section": "amenities", "dataset": name, "status": "ok", "rows": count})

    except Exception as e:
        print(f"  ✗ {table} — {e}")
        ingest_log.append({"section": "amenities", "dataset": name, "status": "error", "error": str(e)})

## C4. Amenities spot checks

In [0]:
spark.sql("SELECT * FROM airbnb_app.raw.amenities_gp_surgeries LIMIT 5").display()

In [0]:
spark.sql("SELECT * FROM airbnb_app.raw.amenities_parks LIMIT 5").display()

In [0]:
spark.sql("SELECT * FROM airbnb_app.raw.amenities_rail_stations LIMIT 5").display()

---
# Part D — Private Rent Data

**Source:** ONS Price Index of Private Rents (PIPR)  
**Granularity:** LAD and region level  
**Coverage:** UK-wide but **Edinburgh is not present** in this dataset  
**Format:** Long format time series from Jan 2015, one row per month per area  

> Edinburgh rental data must be sourced separately from the Scottish Government private sector rent statistics: `https://www.gov.scot/publications/private-sector-rent-statistics-scotland`  

Key columns: `rental_price`, `rental_price_one_bed`, `rental_price_two_bed`, `rental_price_three_bed`, `rental_price_four_or_more_bed`

## D1. Check rent file exists

In [0]:
rent_file_path = f"{RENT_PATH}/{RENT_FILE}"
print("Checking rent file...")
if not check_file(rent_file_path):
    raise FileNotFoundError(f"File not found: {rent_file_path}")

## D2. Ingest rent data to Delta

In [0]:
RENT_TABLE = f"{RAW_DB}.rent_data"
print("Ingesting rent data...")

try:
    df_rent = pd.read_excel(
        rent_file_path,
        sheet_name=RENT_SHEET,
        skiprows=RENT_SKIPROWS,
        dtype=str,
        engine="openpyxl",
    )
    df_rent = df_rent.dropna(how="all")
    df_rent = clean_column_names(df_rent)

    print(f"  Shape: {df_rent.shape}")
    print(f"  Sample columns: {list(df_rent.columns[:6])}")

    count = write_raw_pandas(df_rent, RENT_TABLE, source="ONS Price Index of Private Rents (PIPR)")
    ingest_log.append({"section": "rent", "dataset": "rent_data", "status": "ok", "rows": count})

except Exception as e:
    print(f"  ✗ {RENT_TABLE} — {e}")
    ingest_log.append({"section": "rent", "dataset": "rent_data", "status": "error", "error": str(e)})

## D3. Rent data spot checks

In [0]:
spark.sql("""
    SELECT time_period, area_code, area_name,
           rental_price, rental_price_one_bed,
           rental_price_two_bed, rental_price_three_bed
    FROM airbnb_app.raw.rent_data
    LIMIT 10
""").display()

In [0]:
# Confirm cities present — note Edinburgh will not appear
spark.sql("""
    SELECT DISTINCT area_name, area_code
    FROM airbnb_app.raw.rent_data
    WHERE area_name LIKE '%London%'
       OR area_name LIKE '%Manchester%'
       OR area_name LIKE '%Edinburgh%'
       OR area_name LIKE '%Bristol%'
    ORDER BY area_name
""").display()

In [0]:
# Most recent time period available
spark.sql("""
    SELECT DISTINCT time_period
    FROM airbnb_app.raw.rent_data
    ORDER BY time_period DESC
    LIMIT 5
""").display()

---
## Ingestion Summary

In [0]:
summary = pd.DataFrame(ingest_log)
display(summary)

failures = summary[summary["status"] == "error"]
if not failures.empty:
    raise RuntimeError(
        f"Ingestion failed for:\n{failures[['section', 'dataset', 'error']].to_string()}"
    )

print("\nAll ingestion complete.")

## Notes

**Inside Airbnb**
- Files keep their original Inside Airbnb names — no renaming needed.
- Databricks creates a double city subfolder on upload e.g. `/london/london/` — handled by `resolve_path()`.
- `.csv.gz` files decompressed automatically in A2.

**House prices**
- England and Wales only — Edinburgh requires separate data from Registers of Scotland.
- Raw file ingested in wide format — reshaping to one row per MSOA with most recent price happens in the cleaning notebook.
- All column names lowercased and spaces replaced with underscores for Delta compatibility.

**Amenities**
- GP surgeries and parks: LAD level, England and Wales only — Edinburgh will have no rows.
- Rail stations: MSOA level, England and Wales only.
- `[c]` suppressed values in raw data treated as nulls in the cleaning notebook.
- `clean_column_names()` drops null column names and removes special characters.

**Rent data**
- Edinburgh is not present in the ONS PIPR dataset.
- Scottish rental data available from: `https://www.gov.scot/publications/private-sector-rent-statistics-scotland`
- Cleaning notebook filters to most recent 12 months and pivots to one row per area.

**Next step:** Run `02_clean_airbnb.ipynb` to cast types, select columns, and write to `airbnb_app.clean`.